# REST API example in R

In [3]:
# install.packages("httr2")
library(httr2)

In [5]:
req <- request("https://clinicaltrials.gov/api/v2/studies") |>
  req_url_query(`query.cond` = "diabetes",
                pageSize = 3, format = "json")

req

<httr2_request>
GET https://clinicaltrials.gov/api/v2/studies?query.cond=diabetes&pageSize=3&format=json
Body: empty

In [7]:
resp <- req |> req_perform()
resp

<httr2_response>
GET https://clinicaltrials.gov/api/v2/studies?query.cond=diabetes&pageSize=3&format=json
Status: 200 OK
Content-Type: application/json
Body: In memory (46050 bytes)

In [8]:
resp_status(resp)

[1] 200

### Explore nested fields before constructing a table

In [10]:
body <- resp_body_json(resp, simplifyVector = FALSE)


In [ ]:
body

In [11]:
length(body$studies)

[1] 3

In [12]:
body$studies[[1]]$protocolSection$identificationModule$nctId

[1] "NCT03557541"

In [13]:
body$studies[[1]]$protocolSection$statusModule$overallStatus


[1] "COMPLETED"

In [41]:
# Extract a single field from a study record, returning NA if missing.
field <- function(s, module, name) {
  m <- s$protocolSection[[module]]
  if (is.null(m)) return(NA_character_)
  v <- m[[name]]
  if (is.null(v) || length(v) == 0) return(NA_character_)
  as.character(v[[1]])
}


In [40]:
# title = field(studies[[1]], "identificationModule", "briefTitle")
title = field(studies[[1]], "identificationModule", "briefTsaite")
title

[1] NA

### JSON → DATA FRAME

In [14]:
studies <- body$studies


In [15]:
trials <- do.call(rbind, lapply(studies, function(s) {
  p <- s$protocolSection
  data.frame(
    nct_id = p$identificationModule$nctId,
    title  = p$identificationModule$briefTitle,
    status = p$statusModule$overallStatus
  )
}))

In [17]:
req <- request("https://clinicaltrials.gov/api/v2/studies") |>
  req_url_query(
    `query.cond` = "type 2 diabetes",
    `filter.overallStatus` = "RECRUITING",
    pageSize = 10, format = "json"
  )


In [18]:
req

<httr2_request>
GET https://clinicaltrials.gov/api/v2/studies?query.cond=type%202%20diabetes&filter.overallStatus=RECRUITING&pageSize=10&format=json
Body: empty

In [16]:
print(trials)

       nct_id                                                                                               title    status
1 NCT03557541                                                Sardine-enriched Diet for Prevention Type 2 Diabetes COMPLETED
2 NCT05648383 Promote Health With Digital Tools Among Adults With Type 2 Diabetes/Prediabetes and/or Hypertension COMPLETED
3 NCT06525441                                                    Dietary Protein Sources and Glycemic Variability   UNKNOWN


### Add filters without rewriting the endpoint

In [19]:
resp <- req |> req_perform()

body <- resp_body_json(resp, simplifyVector = FALSE)
studies <- body$studies
trials <- do.call(rbind, lapply(studies, function(s) {
  p <- s$protocolSection
  data.frame(
    nct_id = p$identificationModule$nctId,
    title  = p$identificationModule$briefTitle,
    status = p$statusModule$overallStatus
  )
}))

In [20]:
print(trials)

        nct_id
1  NCT04932928
2  NCT07372677
3  NCT07833930
4  NCT07023172
5  NCT03132129
6  NCT07437157
7  NCT07653477
8  NCT07675499
9  NCT07512934
10 NCT06805786
                                                                                                                                                                    title
1                                                                                 Patient-Driven Lifestyle Modification Using FreeStyle Libre in Type 2 Diabetes Patients
2                                                                                                                           Pocket-X Gel Non-surgical Periodontal Therapy
3                                                                   Incidence and Clinical Features of Breakthrough Pain After Brachial Plexus Block in Diabetic Patients
4                                               Multicenter, Prospective, Observational Study on the Treatment of Type 2 Diabetes With Proline Plus Empagli

### A two-page retrieval pattern

In [32]:
# Get a page of study records from the ClinicalTrials.gov API.
get_page <- function(endpoint = NULL, token = NULL, n = 10,
                     condition = "type 2 diabetes", status = "RECRUITING") {
  req <- request(endpoint) |>
    # Add query parameters for the search condition, status filter, page size, and response format.
    req_url_query(
      `query.cond` = condition,
      `filter.overallStatus` = status,
      pageSize = n,
      format = "json"
    ) |>
    # Set a timeout and retry policy for the request.
    req_timeout(30) |>
    req_retry(max_tries = 3)

  # If a page token is provided, add it to the request.
  if (!is.null(token)) req <- req |> req_url_query(pageToken = token)
  resp <- req |> req_perform()
  cat("HTTP status:", resp_status(resp), "\n")
  resp_body_json(resp, simplifyVector = FALSE)
}



In [34]:
# Convert a list of study records to a data frame with selected fields.
to_table <- function(studies) {
  if (length(studies) == 0) return(data.frame(
    nct_id = character(), title = character(), status = character()))
  do.call(rbind, lapply(studies, function(s) data.frame(
    nct_id = field(s, "identificationModule", "nctId"),
    title = field(s, "identificationModule", "briefTitle"),
    status = field(s, "statusModule", "overallStatus"),
    stringsAsFactors = FALSE
  )))
}

In [35]:
base_url <- "https://clinicaltrials.gov/api/v2/studies"
first <- get_page(endpoint = base_url, token = NULL, n = 10, 
  condition = "type 2 diabetes", status = "RECRUITING")
all_studies <- first$studies

if (!is.null(first$nextPageToken)) {
  second <- get_page(endpoint = base_url, token = first$nextPageToken, n = 10)
  all_studies <- c(all_studies, second$studies)
}
trials <- to_table(all_studies)


HTTP status: 200 
HTTP status: 200 


In [36]:
print(trials)

        nct_id
1  NCT04932928
2  NCT07372677
3  NCT07833930
4  NCT07023172
5  NCT03132129
6  NCT07437157
7  NCT07653477
8  NCT07675499
9  NCT07512934
10 NCT06805786
11 NCT07376486
12 NCT06993454
13 NCT06118931
14 NCT05893927
15 NCT06862180
16 NCT06694155
17 NCT04485871
18 NCT01055093
19 NCT06478186
20 NCT06921356
                                                                                                                                                                    title
1                                                                                 Patient-Driven Lifestyle Modification Using FreeStyle Libre in Type 2 Diabetes Patients
2                                                                                                                           Pocket-X Gel Non-surgical Periodontal Therapy
3                                                                   Incidence and Clinical Features of Breakthrough Pain After Brachial Plexus Block in Diabetic Patients
4    